# Localisation des hôpitaux (ville, département, région d'appartenance)

OBJECTIF : 
L'objectif est de transformer les coordonnées géographiques relatives aux hôpitaux en ville d'appartenance, département d'appartenance et région d'appartenance.
Pour cela, nous mobilisons des données géographiques relatives aux établissements hospitaliers ainsi que les contours administratifs des communes françaises.
La méthode repose sur une jointure spatiale réalisée avec la bibliothèque GeoPandas, permettant d'identifier dans quelle commune se situe chaque établissement.

SOURCES ET REFERENCES

- Données hospitalières : data.gouv.fr
- Contours communaux : Cartiflette
- Référence méthodologique : Lino Galiana, *Introduction aux données spatiales avec GeoPandas*.


In [9]:
%pip install requests
%pip install pandas
%pip install geopandas
%pip install cartiflette


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.3/33.3 MB 89.0 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 94.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 84.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [geopandas]/4 [geopandas]

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 85.6 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 94.9 MB/s 

In [10]:
## 1. IMPORTATION DES PACKAGES
# Gestion des fichiers et téléchargement des données
import os
import requests

# Manipulation des données
import pandas as pd
import geopandas as gpd

# Traitement des données géographiques
from shapely.wkt import loads
from cartiflette import carti_download

Les bibliothèques mobilisées permettent :
- le téléchargement des données depuis Internet ;
- la manipulation et l'organisation des données tabulaires ;
- le traitement des données géographiques ;
- le téléchargement des contours administratifs.

## 2. Importation et exploration des données hospitalières
Les données relatives aux établissements hospitaliers sont téléchargées depuis la plateforme data.gouv.fr.

Une première exploration permet de vérifier la structure de la base, les variables disponibles et la qualité des informations de localisation.

In [11]:
#Fonction de téléchargement
def telecharger(url, chemin):
    """Télécharge url et l'enregistre dans chemin (sans retélécharger si le fichier existe)."""
    if os.path.exists(chemin):
        return
    os.makedirs(os.path.dirname(chemin), exist_ok=True)
    reponse = requests.get(url, timeout=60)
    reponse.raise_for_status()                 # erreur claire si le lien est mort
    with open(chemin, "wb") as f:
        f.write(reponse.content)

In [12]:
#Téléchargement et lecture
URL_APL = "https://www.data.gouv.fr/api/1/datasets/r/c41a1919-fbd7-4d19-a8df-1719bef8b14a"
telecharger(URL_APL, "data/raw/hospitals_point.csv")

df = pd.read_csv("data/raw/hospitals_point.csv")
df

,FID,osm_id,amenity,name,short_name,official_name,alt_name,old_name,operator,operator-type,...,description,opening_hours,source,note,osm_type,osm_version,osm_timestamp,healthcare-speciality,the_geom,osm_original_geom
0,hospitals_point.199789928,199789928,hospital,Établissement Public de Santé de Fresnes,EPSNF,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,extraction vectorielle v1 cadastre-dgi-fr sour...,Prison Hôpital,way,2,2020-06-09T10:14:16Z,NaN,POINT (258310.14044964363 6235141.052489291),SRID=3857;POLYGON((258275.23376428554 6235067....
1,hospitals_point.575489561,575489561,hospital,Centre Hospitalier du Penthièvre & du Poudouvre,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,way,6,2026-05-20T08:45:54Z,NaN,POINT (-324498.6131004042 6174952.219867987),SRID=3857;POLYGON((-324714.2458295185 6174891....
2,hospitals_point.782668290,782668290,hospital,Clinique Pech du Soleil,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,INSEE - 2026-05,NaN,way,3,2026-05-27T16:35:31Z,NaN,POINT (361639.85189134383 5367943.485850866),SRID=3857;POLYGON((361575.2127810663 5367959.6...
3,hospitals_point.38072526,38072526,hospital,Hôpital Bichat-Claude Bernard,NaN,NaN,Hopital Xavier Bichat,NaN,Assistance publique - Hôpitaux de Paris,public,...,NaN,NaN,NaN,NaN,way,18,2023-10-13T14:15:47Z,intensive,POINT (259592.22219624888 6257717.90984779),SRID=3857;POLYGON((259309.3250420606 6257581.0...
4,hospitals_point.106491473,106491473,hospital,Unité de Santé Mentale pour Adultes,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,cadastre-dgi-fr source : Direction Générale de...,NaN,way,4,2019-12-23T23:21:24Z,NaN,POINT (209790.92290423455 6613791.652966853),SRID=3857;POLYGON((209736.71864513436 6613827....
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2501,hospitals_point.1075425737,1075425737,hospital,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,way,2,2024-08-23T11:01:06Z,NaN,POINT (-125955.87209774728 5582755.398745799),SRID=3857;POLYGON((-125975.70363162518 5582751...
2502,hospitals_point.283415613,283415613,hospital,Clinique Rougemont,NaN,NaN,NaN,NaN,Inicea,NaN,...,NaN,Mo-Su 09:00-18:00,cadastre-dgi-fr source : Direction Générale de...,NaN,way,14,2026-08-11T10:22:07Z,physiatry,POINT (17391.457057163512 6110212.548394944),SRID=3857;POLYGON((17337.287114362203 6110237....
2503,hospitals_point.699215560,699215560,hospital,Clinique de l'Étang de l'Olivier,NaN,NaN,Clinique privée d'Istres,NaN,Almaviva Santé,private,...,NaN,NaN,NaN,NaN,way,1,2019-06-23T13:34:31Z,NaN,POINT (555714.7492888186 5389626.65922673),SRID=3857;POLYGON((555590.60256799 5389634.877...
2504,hospitals_point.226685761,226685761,hospital,Institut Paoli-Calmettes,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,way,18,2026-06-29T14:47:45Z,intensive,POINT (602228.8049932518 5351990.283724458),SRID=3857;POLYGON((601792.477047594 5351872.06...


Nous utiliserons la colonne the_geom pour localiser les hôpitaux. On constate qu'il y a beaucoup de variables intéressantes, qui ne sont toutefois pas toujours parfaitement renseignées. Nous allons voir que c'est notamment le cas de la vairable public/privé (operator type).

In [13]:
#Exploration

#Vérification des valeurs manquantes pour l'appartenance au public ou au privé : 
print("Pourcentage de NA public/privé :", df['operator-type'].isna().mean() * 100)
#L'information n'est pas renseignée pour prêt de 65% des hôpitaux

#Par contre d'autres informations sont systématiquement remplies, c'est le cas de la localisation, l'information qui nous intéresse : 
print("Pourcentage de NA localisation :", df['the_geom'].isna().mean() * 100)


Pourcentage de NA public/privé : 64.44533120510773
Pourcentage de NA localisation : 0.0
